# Caso Práctico: Regresión Lineal

**¿Qué es regresión?**


Es un proceso que permite la estimación de las relaciones funcionales entre variables. En el campo de aprendizaje automático es ampliamente utilizado para la predicción del valor de una variable dependiente, a partir de una o más variables explicativas. El modelo de regresión puede ser lineal o no lineal.


**¿Qué preguntas se pueden respodner mediante regresiones?**

- ¿Cómo varían el volumen de ventas cuando subimos los precios?
- ¿Afecta el nombre de un producto a su contratación?
- ¿Cuántos clientes contratarán hoy un plan de pensiones?


**¿Cuándo usar regresión lineal?**

- Cuando queremos predecir el valor de una variable a partir de otras.
- Si queremos explicar o entender la relación entre dos o más atributos.

---

## Regresión lineal simple

Los problemas de regresión tienen una estructura común: una variable respuesta ($y$) que puede ser expresada como combinación de una o más variables independientes ($x_i$), llamadas covariables o predictores. El algoritmo de regresión intenta construir un modelo que exprese la variable respuesta en función de las covariables, como:

$$y = a_1x_1 + a_2x_2 + \ldots + a_nx_n$$

donde $a_i$ son los parámetros del modelo, llamados coeficientes. El modelo se ajusta mediante Mínimos Cuadrados Ordinarios (OLS en inglés), donde los coeficientes son elegidos para minimizar el cuadrado de la distancia (vertical) entre los valores predichos y los reales.

*Ejemplo*. Para entender las funcionalidades de Python para regresión vamos a construir un modelo sencillo de regresión a partir de datos sintéticos usando varias librerías científicas de Python.  

Para ello vamos a generar 300 valores aleatorios de una distribución Gaussiana, los multiplicamos por unos coeficientes que les den una forma aproximadamente lineal.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
#  Números decimales muestreados de una distribución Normal estándar
datos = np.random.randn(300, 2)
pesos = np.array([[0.6, .4], [.4, 0.6]])
datos = np.dot(datos, pesos)

x = datos[:,0]
y = datos[:,1]

In [ ]:
plt.plot(x, y, "ro", c='orange', alpha=0.3)

In [ ]:
type(x)

### Scikit-learn

Scikit-learn posee una interfaz orientada a objetos basada en el concepto de un *Estimador*

El método <code>Estimator.fit</code> establece el estado del estimador de acuerdo a los datos de entrenamiento. Usualmente estos datos están representdos por un `numpy.array` bidimensional $x$ con dimensiones <code>(n_muestras, n_predictores)</code> que contiene la matriz de características, y un `numpy.array` unidimensional que contiene los valores de la variable de respuesta $y$.

In [ ]:
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

In [ ]:
x.shape

In [ ]:
x = x.reshape(-1,1)

In [ ]:
x.shape

In [ ]:
x

In [ ]:
regr = LinearRegression(fit_intercept=True)
regr.fit(x, y)
y_hat = regr.predict(x)

print('Coeficientes: ', regr.intercept_, regr.coef_)
print('Error medio: ', mean_absolute_error(y, y_hat))

plt.scatter(x, y, c='orange', marker='o', alpha=0.5)
plt.plot(x, y_hat, alpha=0.5)

El método ``Estimator.predict`` permite hacer predicciones. En el caso de regresión, este método devuelve los valores predichos por el modelo.

Existe también un objeto `Estimator` especial llamado `Transformer` que permite realizar transformaciones sobre los datos. En el caso de regresión, una transformación adecuada es la de normalizar los predictores, tal que tengan media cero y desviación típica 1 con ``sklearn.preprocessing.StandardScaler``

---

# Regresión lineal múltiple

## Caso: Predicción de Picos de Intensidad de impactos en un Molino SAG

### 1. Importación de Librerías

In [ ]:
import matplotlib #.pyplot as plt1
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
import scipy as sp
import seaborn as sns
import scipy.stats as stats
import statsmodels.api as sm
import statsmodels.tsa.api as smt
import warnings
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

: 

In [ ]:
warnings.filterwarnings("ignore")
%matplotlib inline

### 2. Importación de Datos

In [ ]:
df = pd.read_csv('01 dbCasoPracticoMolino.csv',delimiter=';')
df.head()

In [ ]:
df.shape

In [ ]:
df.groupby('Intensidad de impactos').count()

### 3. Limpieza de Datos

In [ ]:
titles=list(df)

In [ ]:
titles

In [ ]:
##Tratamiento de datos invalidos
#-------------------------------------------

for col in titles:
    #Eliminar filas cuyo valor en columna tenga un dato vacio
    df=df.dropna(subset=[col])
    #Eliminar valores invalidos
    df = df.drop(df[(df[col]=='Bad') | (df[col]=='Shutdown') | (df[col]=='I/O Timeout')| (df[col]=='[-11059] No Good Data For Calculation') | (df[col]==' ')].index)

#Filtro Sag encendido
df = df.drop(df[df['SAG Run Sts']=='Stop'].index)
df = df.drop(['SAG Run Sts'], axis=1)
df.isnull().sum(axis=0)

df = df.drop(['Fecha'], axis=1) # Igual a: df.drop(['Fecha'], axis='columns', inplace=True)
df=df.astype(float).round(2)

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
#1. Analisis de Correlaciones de Pearson
corr_mat=df.corr(method='pearson')
plt.figure(figsize=(10,8))
sns.heatmap(corr_mat,vmax=1,square=True,annot=True,center= 0,cmap='coolwarm') # other params: linewidths=3, linecolor='black'

### 4. Modelamiento Predictivo

In [ ]:
#col = lista
col = df.columns
col

In [ ]:
# Target: Variable que intentamos pronosticar
target_col = "Intensidad de impactos"

# Definicion de variables de estudio
X = df.loc[:, col != target_col]

y = df.loc[:, target_col]

#Realizamos una selección de muestra ENTRENAMIENTO Y VALIDACION aletoria y comparables muestras, con
#fines de validar el aprendizaje de nuestro modelo
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.30,
                                                    random_state=42)

In [ ]:
def indicadores_regresion(y_test,y_pred):
    # Calculamos el Root Mean Square Error: Suma(Y_reales-Y_predichos)^2/n
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)

    # Definimos y calculamos el MAPE (mean_absolute_percentage_error)
    y_test, y_pred = np.array(y_test), np.array(y_pred)
    mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100

    return rmse,mape

#### 4.1. Modelo de Regresion Lineal

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
# Entrenamiento del modelo
model_regression_lineal = LinearRegression()
model_regression_lineal.fit(X_train, y_train)

In [ ]:
# Predecir el conjunto de test
y_pred = model_regression_lineal.predict(X_test)

In [ ]:
residual = y_test - y_pred
residual

In [ ]:

# Mostrar indicadores de precision
rmse, mape = indicadores_regresion(y_test,y_pred)
print(f'RMSE: {rmse} - MAPE: {mape}')

# Gráfico: Validación del Residual como una Distribución Normal
ax = sns.distplot(residual)

# Gráfico: Validación de Homocedasticidad
fig, ax = plt.subplots(figsize=(10,5))
ax.scatter(y_pred, residual)

In [ ]:
# regression coefficients and R-squared value of model
print(model_regression_lineal.intercept_, model_regression_lineal.coef_, model_regression_lineal.score(X_train, y_train))

In [ ]:
ecuacion_regresion = dict(zip(X_train.columns, model_regression_lineal.coef_))
ecuacion_regresion['INTERCEPTO'] = model_regression_lineal.intercept_
ecuacion_regresion